# Auditing whether a model *causally uses* a governing law

A concise, self-contained walkthrough. One idea, three settings:

1. **Abstract BP** — the two components are plain numbers (PTT, calibration).
2. **BP simulator** — the pulse component becomes an actual waveform.
3. **SpO₂ simulator** — two optical channels, where the law is an *invariance*.

Throughout, the point is the same: a model can be **accurate** and have the answer
**decodable** inside it, yet still not compute the answer *through* the law. The
governing equation — which we know — lets us catch that on a **frozen** model,
with no extra labels.

In [1]:
import collections
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0); np.random.seed(0)

# A tiny container: xi, xj are the two components fed to the model;
# a, b are the raw scalars we keep so the equation can act as an oracle.
D = collections.namedtuple("D", "xi xj y a b")

## Part 1 — Abstract BP

**Governing law** (Moens–Korteweg inversion, in scaled units; real BP = 100 + 40·y):

$$ y = 2\ln(L/\text{PTT}) - \ln(E_0) $$

PTT is the pulse signal; **E₀ is the per-subject calibration**. The answer needs
*both* — the same PTT with a different E₀ means a different BP. That is the
identifiability boundary the whole method turns on.

In [2]:
L = 0.22
def bp_true(ptt, e0):  return 2*np.log(L/ptt) - np.log(e0)   # true law: uses BOTH components
def bp_wrong(ptt, e0): return 2*np.log(L/ptt) + np.log(e0)   # unfaithful: calibration sign flipped

def bp_data(n, vary_e0=True, wrong=False, seed=0):
    r = np.random.default_rng(seed)
    ptt = r.uniform(.18, .28, n)                              # component i (the signal)
    e0  = r.uniform(.5, 2., n) if vary_e0 else np.ones(n)     # component j (calibration)
    y   = (bp_wrong if wrong else bp_true)(ptt, e0)
    t = lambda z: torch.tensor(z, dtype=torch.float32)
    return D(t(ptt).reshape(-1,1), t(e0).reshape(-1,1), t(y), ptt, e0)

**The model.** A deliberately simple design: encode each component separately, so
the hidden state is `h = [h_i ; h_j]`. Component j (calibration) lives in the
second half — which makes the audit's "swap" a one-liner. (In a real transformer
the components are *not* axis-aligned; you'd learn a rotation to find the
subspace — that is DAS. Here we skip that so the idea is visible.)

In [3]:
class TwoPart(nn.Module):
    def __init__(self, din_i=1, din_j=1, w=16):
        super().__init__()
        self.ei = nn.Sequential(nn.Linear(din_i,32), nn.ReLU(), nn.Linear(32,w))  # encodes component i
        self.ej = nn.Sequential(nn.Linear(din_j,32), nn.ReLU(), nn.Linear(32,w))  # encodes component j
        self.head = nn.Sequential(nn.Linear(2*w,32), nn.ReLU(), nn.Linear(32,1))
        self.w = w
    def state(self, xi, xj): return torch.cat([self.ei(xi), self.ej(xj)], 1)       # h = [h_i ; h_j]
    def forward(self, xi, xj): return self.head(self.state(xi, xj)).squeeze(1)

def train(m, d, ep=400):
    opt = torch.optim.Adam(m.parameters(), 3e-3)
    for _ in range(ep):
        opt.zero_grad(); (((m(d.xi, d.xj)-d.y)**2).mean()).backward(); opt.step()
    return m

**The audits** (all run on a frozen model, shared by every part):

- `mse` — accuracy on the data you have.
- `probe` — can a *linear* readout recover the answer from the state? (presence ≠ use)
- `grad_j` — signed `∂output/∂(component j)`: does it use j, and in which direction?
- `interchange` — **the causal audit**: overwrite the component-j half of the state
  with another input's, re-run the head, and check the output becomes what the
  *equation* predicts.

In [4]:
def mse(m, d):
    with torch.no_grad(): return float(((m(d.xi, d.xj)-d.y)**2).mean())

def probe(m, d):                                    # linear decodability of the answer from the state
    with torch.no_grad(): H = m.state(d.xi, d.xj).numpy()
    H = np.c_[H, np.ones(len(H))]; y = d.y.numpy()
    coef, *_ = np.linalg.lstsq(H, y, rcond=None)
    return float(1 - ((y - H@coef)**2).mean()/y.var())

def grad_j(m, d):                                   # signed sensitivity to component j
    xj = d.xj.clone().requires_grad_(True)
    g, = torch.autograd.grad(m(d.xi, xj).sum(), xj)
    return float(g.mean())

def interchange(m, base, src, oracle):              # swap the component-j half of the state
    with torch.no_grad():
        hb, hs = m.state(base.xi, base.xj), m.state(src.xi, src.xj)
        w = m.w
        hp = torch.cat([hb[:, :w], hs[:, w:]], 1)   # keep base's i, take source's j
        yh = m.head(hp).squeeze(1).numpy()
    tgt = oracle(base, src)                          # what the EQUATION says the answer should become
    return float(1 - ((yh - tgt)**2).mean()/tgt.var())

def shuffled(d):
    i = np.random.permutation(len(d.y))
    return D(d.xi[i], d.xj[i], d.y[i], d.a[i], d.b[i])

def report(models, val, ood, oracle):
    src = shuffled(ood)
    print(f"{'model':11s}{'valMSE':>8}{'probeR2':>9}{'grad_j':>8}{'interch':>9}{'oodMSE':>8}")
    for name, m in models:
        print(f"{name:11s}{mse(m,val):>8.3f}{probe(m,val):>9.2f}{grad_j(m,val):>+8.2f}"
              f"{interchange(m,ood,src,oracle):>+9.2f}{mse(m,ood):>8.3f}")

bp_oracle = lambda base, src: bp_true(base.a, src.b)   # counterfactual: base's PTT, source's E0

**Three models, same architecture and task.** `law` sees calibration vary (true
law); `unfaithful` is trained on the wrong-sign law (uses j, wrong way);
`shortcut` sees calibration fixed (ignores j). Validation has E₀≈1 (the cohort you
have); OOD lets E₀ vary (reality).

In [5]:
print("PART 1  abstract BP  (components are scalars)")
law = train(TwoPart(1), bp_data(3000, True,  False, 1))
unf = train(TwoPart(1), bp_data(3000, True,  True,  1))
sho = train(TwoPart(1), bp_data(3000, False, False, 1))
report([("law",law),("unfaithful",unf),("shortcut",sho)],
       bp_data(2000, False, False, 2), bp_data(2000, True, False, 3), bp_oracle)

PART 1  abstract BP  (components are scalars)


model        valMSE  probeR2  grad_j  interch  oodMSE
law           0.000     1.00   -0.94    +1.00   0.000
unfaithful    0.000     1.00   +1.00    -2.18   0.703
shortcut      0.002     1.00   -0.11    +0.23   0.169


Read the table: **valMSE and probeR2 are identical** for all three — accuracy and
decodability cannot tell them apart. `grad_j` flags the shortcut (≈0) but is
*fooled* by the unfaithful model (same magnitude, wrong sign). Only **interch**
isolates the law model (≈ +1), and **oodMSE** confirms it afterward.

## Part 2 — A simple BP simulator

Now the pulse component is an **actual two-channel waveform**: a proximal pulse and
a distal pulse whose *lag* encodes PTT. The model must read the lag out of the
signal. Only the encoder for component i changes — everything else is identical.

In [6]:
T = 48; tg = np.linspace(0, 1, T)
def bp_wave(n, vary_e0=True, wrong=False, seed=0):
    r = np.random.default_rng(seed)
    ptt = r.uniform(.18, .28, n)
    e0  = r.uniform(.5, 2., n) if vary_e0 else np.ones(n)
    prox = np.tile(np.exp(-((tg-0.3)**2)/(2*0.05**2)), (n,1))          # proximal pulse (fixed)
    center = 0.35 + 2.0*(ptt-0.18)                                     # distal center = lag ∝ PTT
    dist = np.exp(-((tg[None,:]-center[:,None])**2)/(2*0.05**2))
    sig = np.stack([prox, dist], 1).reshape(n, 2*T).astype("float32")  # 2 channels, flattened
    y = (bp_wrong if wrong else bp_true)(ptt, e0)
    t = lambda z: torch.tensor(z, dtype=torch.float32)
    return D(t(sig), t(e0).reshape(-1,1), t(y), ptt, e0)

print("PART 2  BP simulator  (component i is a waveform; the model must read the lag)")
law = train(TwoPart(2*T), bp_wave(3000, True,  False, 1))
unf = train(TwoPart(2*T), bp_wave(3000, True,  True,  1))
sho = train(TwoPart(2*T), bp_wave(3000, False, False, 1))
report([("law",law),("unfaithful",unf),("shortcut",sho)],
       bp_wave(2000, False, False, 2), bp_wave(2000, True, False, 3), bp_oracle)

PART 2  BP simulator  (component i is a waveform; the model must read the lag)


model        valMSE  probeR2  grad_j  interch  oodMSE
law           0.000     1.00   -1.00    +1.00   0.000
unfaithful    0.000     1.00   +1.08    -2.21   0.704
shortcut      0.000     1.00   -0.06    +0.28   0.158


Same verdict, now on a real signal: the model extracts PTT from the waveform, yet
the audit still separates faithful from shortcut/unfaithful where accuracy and
probing cannot.

## Part 3 — Extend to SpO₂

Different law, different shape. SpO₂ comes from the **ratio-of-ratios**
`R = (AC_red/DC_red)/(AC_ir/DC_ir)`, which is engineered to be **invariant to
perfusion**. So the identifiability boundary here is an *invariance*, and the
natural audit is: *same SpO₂ at two different perfusions must give the same
answer.* (A channel-swap interchange would create a physically impossible
mismatched-perfusion input — a nice reminder that the audit should match the law's
structure.)

- **faithful** — trained with perfusion varying → must use the invariant ratio.
- **shortcut** — trained with perfusion fixed → reads one raw channel, confounds
  SpO₂ with perfusion.

In [7]:
def spo2_data(n, vary_perf=True, seed=0):
    r = np.random.default_rng(seed)
    spo2 = r.uniform(.85, 1., n)                              # truth
    perf = r.uniform(.5, 1.5, n) if vary_perf else np.ones(n) # perfusion (a nuisance the ratio removes)
    a_red = 0.5 + 0.5*(1-spo2)                                # per-channel AC/DC (perfusion-free)
    a_ir  = 0.5 + 0.5*spo2
    xi = np.stack([perf*a_red, perf], 1).astype("float32")   # red channel = [AC, DC]  (component i)
    xj = np.stack([perf*a_ir,  perf], 1).astype("float32")   # ir  channel = [AC, DC]  (component j)
    t = lambda z: torch.tensor(z, dtype=torch.float32)
    return D(t(xi), t(xj), t(spo2), a_red, a_ir)

# the physics audit: same SpO2, two perfusions -> a faithful model gives the same answer
def invariance(m, seed=10, n=2000):
    r = np.random.default_rng(seed)
    spo2 = r.uniform(.85, 1., n)
    a_red = 0.5 + 0.5*(1-spo2); a_ir = 0.5 + 0.5*spo2
    def build(perf):
        xi = np.stack([perf*a_red, perf], 1).astype("float32")
        xj = np.stack([perf*a_ir,  perf], 1).astype("float32")
        return torch.tensor(xi), torch.tensor(xj)
    xi1, xj1 = build(r.uniform(.5, 1.5, n))
    xi2, xj2 = build(r.uniform(.5, 1.5, n))
    with torch.no_grad():
        return float((m(xi1, xj1) - m(xi2, xj2)).abs().mean())

print("PART 3  SpO2 simulator  (faithful uses the perfusion-invariant ratio)")
faithful = train(TwoPart(2, 2), spo2_data(3000, True,  1))
shortcut = train(TwoPart(2, 2), spo2_data(3000, False, 1))
val = spo2_data(2000, False, 2); ood = spo2_data(2000, True, 3)
print(f"{'model':11s}{'valMSE':>9}{'probeR2':>9}{'invariance':>12}{'oodMSE':>9}")
for name, m in [("faithful", faithful), ("shortcut", shortcut)]:
    print(f"{name:11s}{mse(m,val):>9.4f}{probe(m,val):>9.2f}{invariance(m):>12.3f}{mse(m,ood):>9.4f}")

PART 3  SpO2 simulator  (faithful uses the perfusion-invariant ratio)


model         valMSE  probeR2  invariance   oodMSE
faithful      0.0001     1.00       0.009   0.0002
shortcut      0.0016     1.00       0.170   0.0249


Again validation and probe are identical, but the **invariance** audit exposes the
shortcut (it changes its SpO₂ when only perfusion changes), and OOD confirms it.

### The one takeaway

The governing law tells you *what to check* and *how to grade it*, on a frozen
model, with no extra labels. The instrument adapts to the law's identifiability
structure — a **calibration swap** for BP, an **invariance test** for SpO₂ — but
the principle is one line: *does the model compute the answer through the law, or
merely land on it?*